In [ ]:
import numpy as np
import qnm
import sxs
import h5py
from scipy import interpolate
import pickle
import pandas as pd

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal
from itertools import combinations_with_replacement

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.rcParams['figure.max_open_warning'] = 0

plt.rcParams['text.usetex']        = True

# plt.rcParams['mathtext.fontset']  = 'stix'
# plt.rcParams['font.family']       = 'STIXGeneral'

plt.rcParams['font.size']         = 14
plt.rcParams['axes.linewidth']    = 1
plt.rcParams['axes.labelsize']    = plt.rcParams['font.size']
plt.rcParams['axes.titlesize']    = 1.5*plt.rcParams['font.size']
plt.rcParams['legend.fontsize']   = plt.rcParams['font.size']
plt.rcParams['xtick.labelsize']   = plt.rcParams['font.size']
plt.rcParams['ytick.labelsize']   = plt.rcParams['font.size']
plt.rcParams['xtick.major.size']  = 3
plt.rcParams['xtick.minor.size']  = 3
plt.rcParams['xtick.major.width'] = 1
plt.rcParams['xtick.minor.width'] = 1
plt.rcParams['ytick.major.size']  = 3
plt.rcParams['ytick.minor.size']  = 3
plt.rcParams['ytick.major.width'] = 1
plt.rcParams['ytick.minor.width'] = 1

plt.rcParams['legend.frameon']             = False
plt.rcParams['legend.loc']                 = 'center left'
plt.rcParams['contour.negative_linestyle'] = 'solid'

In [ ]:
def wfs(res, res_bayes, res_qc, label, NR_re=None, NR_im=None, t_NR=None):
    
    omega, _, _ = qnm.modes_cache(s=-2, l=2, m=2, n=0)(a=np.abs(res[0].af))
    f_rd_fundamental = (np.real(omega) / res[0].Mf) / (2.0 * np.pi)
    tau_rd_fundamental = -1.0 / (np.imag(omega)) * res[0].Mf

    tau = res[0].t_NR - res[0].t_peak
    tau_cut = res[0].t_NR_cut - res[0].t_peak

    NR_phi = res[0].NR_phi
    NR_r, NR_i = res[0].NR_r, res[0].NR_i
    NR_f = res[0].NR_freq

    NR_amp = res[0].NR_amp
    NR_r, NR_i = res[0].NR_r, res[0].NR_i

    nc_re, nc_im = res[2].wf_r, res[2].wf_i
    bayes_re, bayes_im = res_bayes[2].wf_r, res_bayes[2].wf_i
    qc_re, qc_im = res_qc[2].wf_r, res_qc[2].wf_i

    bayes_amp = np.sqrt(bayes_re**2 + bayes_im**2)
    qc_amp = np.sqrt(qc_re**2 + qc_im**2)
    nc_amp = np.sqrt(nc_re**2 + nc_im**2)

    bayes_phase = np.unwrap(np.angle(bayes_re - 1j*bayes_im))
    qc_phase = np.unwrap(np.angle(qc_re - 1j*qc_im))
    nc_phase = np.unwrap(np.angle(nc_re - 1j*nc_im))

    if len(bayes_phase) != len(tau_cut):
        bayes_freq = np.gradient(bayes_phase[:-1], tau_cut)/(2.0*np.pi)
    else:
        bayes_freq = np.gradient(bayes_phase, tau_cut)/(2.0*np.pi)
    if len(qc_phase) != len(tau_cut):
        qc_freq = np.gradient(qc_phase[:-1], tau_cut)/(2.0*np.pi)
    else:
        qc_freq = np.gradient(qc_phase, tau_cut)/(2.0*np.pi)
    nc_freq = np.gradient(nc_phase, tau_cut)/(2.0*np.pi)

    label_data = 'h_{22}'

    COL = {
        'NR': 'k',
        'Global RatExp': '#cc0033',
        'Global HypTan': 'royalblue',
        't_peak': 'royalblue',
        'f_overt': 'darkorange',
        'f_ring': 'forestgreen',
    }

    alpha_std = 1
    alpha_med = 0.8
    alpha_low = 0.9

    LS = {
        't': '--',
        'f': '--',
        'Least Squares': '--',
        'Bayesian': '-',
    }

    LW = {
        'small': 1.25,
        'medium': 1.2,
        'std': 1.8,
        'large': 2.5,
    }

    FONTS = {
        'legend': 18,
        'labels': 23,
    }

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    ax1, ax2, ax3, ax4 = axes.flat

    plot_std = dict(c=COL['NR'], lw=LW['std'], alpha=alpha_std, ls='-')
    vline_peak = dict(lw=LW['small'], alpha=alpha_std, ls=LS['t'])

    ax1.plot(tau, NR_r, **plot_std)
    if len(qc_phase) != len(tau_cut):
        ax1.plot(tau_cut, qc_re[:-1], c=COL['Global HypTan'], lw=LW['large'], alpha=alpha_med, linestyle=LS['Bayesian'])
    else:
        ax1.plot(tau_cut, qc_re, c=COL['Global HypTan'], lw=LW['large'], alpha=alpha_med, linestyle=LS['Bayesian'])
    if len(bayes_phase) != len(tau_cut):
        ax1.plot(tau_cut, bayes_re[:-1], c=COL['Global RatExp'], lw=LW['large'], alpha=alpha_low, linestyle=LS['Bayesian'])
    else:
        ax1.plot(tau_cut, bayes_re, c=COL['Global RatExp'], lw=LW['large'], alpha=alpha_low, linestyle=LS['Bayesian'])
    ax1.plot(tau_cut, nc_re, c=COL['Global RatExp'], lw=LW['large'], alpha=alpha_std, linestyle=LS['Least Squares'])
    ax1.axvline(0.0, c=COL['t_peak'], **vline_peak)
    ax1.set_ylabel(r'$\mathit{Re}(%s)$' % (label_data,), fontsize=FONTS['labels'])

    NR_r_cut_nc = res[0].NR_r_cut
    NR_r_cut_qc = res_bayes[0].NR_r_cut
    NR_r_cut_qc0 = res_qc[0].NR_r_cut

    ax3.plot(tau_cut, (qc_re - NR_r_cut_qc0), c=COL['Global HypTan'], lw=LW['large'], alpha=alpha_med, linestyle=LS['Bayesian'])
    ax3.plot(tau_cut, (bayes_re - NR_r_cut_qc), c=COL['Global RatExp'], lw=LW['large'], alpha=alpha_low, linestyle=LS['Bayesian'])
    ax3.plot(tau_cut, (nc_re - NR_r_cut_nc), c=COL['Global RatExp'], lw=LW['large'], alpha=alpha_std, linestyle=LS['Least Squares'])
    ax3.axvline(0.0, c=COL['t_peak'], **vline_peak)
    ax3.set_ylabel(r'$\delta\mathit{Re}(%s)$' % (label_data,), fontsize=FONTS['labels'])
    ax3.set_xlabel(r'$t - t_{\rm{ref}} \, [M]$', fontsize=FONTS['labels'])
    ax3.axhspan(ymin=-1e-3, ymax=1e-3, color='gray', alpha=0.3)

    ax2.semilogy(tau, NR_amp * np.exp((tau) / tau_rd_fundamental), c=COL['NR'], lw=LW['std'], alpha=alpha_std, ls='-')
    ax2.semilogy(tau_cut, qc_amp * np.exp((tau_cut / tau_rd_fundamental)), c=COL['Global HypTan'], lw=LW['large'], alpha=alpha_med, ls=LS['Bayesian'])
    ax2.semilogy(tau_cut, bayes_amp * np.exp((tau_cut / tau_rd_fundamental)), c=COL['Global RatExp'], lw=LW['large'], alpha=alpha_low, ls=LS['Bayesian'])
    ax2.semilogy(tau_cut, nc_amp * np.exp((tau_cut / tau_rd_fundamental)), c=COL['Global RatExp'], lw=LW['large'], alpha=alpha_std, ls=LS['Least Squares'])
    ax2.axvline(0.0, c=COL['t_peak'], **vline_peak)
    ax2.set_ylabel(r'$\mathit{A_{22}(t)} \cdot \rm{e}^{t/\tau_{22}}$', fontsize=FONTS['labels'])
    ax2.set_ylim(5e-2, 2e0)

    ax4.plot(tau, NR_f, c=COL['NR'], lw=LW['std'], alpha=alpha_std, ls='-')
    ax4.plot(tau_cut, qc_freq, c=COL['Global HypTan'], lw=LW['large'], alpha=alpha_med, ls=LS['Bayesian'])
    ax4.plot(tau_cut, bayes_freq, c=COL['Global RatExp'], lw=LW['large'], alpha=alpha_low, ls=LS['Bayesian'])
    ax4.plot(tau_cut, nc_freq, c=COL['Global RatExp'], lw=LW['large'], alpha=alpha_std, ls=LS['Least Squares'])
    ax4.axhline(f_rd_fundamental, c=COL['f_ring'], lw=LW['small'], alpha=alpha_std, ls=LS['f'])
    ax4.axvline(0.0, c=COL['t_peak'], **vline_peak)
    ax4.set_xlabel(r'$t - t_{\rm{ref}} \, [M]$', fontsize=FONTS['labels'])
    ax4.set_ylabel(r'$\mathit{f_{22}\,(t)}$', fontsize=FONTS['labels'])
    ax4.set_ylim(0, f_rd_fundamental*1.5)

    plt.rcParams['legend.frameon'] = True
        
    custom_lines_1 = [
        Line2D([0], [0], color=COL['NR'], lw=LW['std'], label=r'$\mathrm{NR}$'),
        Line2D([0], [0], color=COL['Global HypTan'], lw=LW['large'], label=r'$\it{HypTan}$'),
        Line2D([0], [0], color=COL['Global RatExp'], lw=LW['large'], label=r'$\it{RatExp}$')
    ]
    ax1.legend(handles=custom_lines_1, loc='lower right', fontsize=FONTS['legend'], shadow=True)

    ax2.legend(handles=[Line2D([0], [0], color=COL['t_peak'], lw=LW['small'], ls=LS['t'], label=r'$t_{\rm peak}$')], loc='lower right', fontsize=FONTS['legend'], shadow=True)

    custom_lines_2 = [
        Line2D([0], [0], color='gray', lw=LW['large'], ls=LS['Least Squares'], label='Least Squares'),
        Line2D([0], [0], color='gray', lw=LW['large'], ls=LS['Bayesian'], label='Bayesian')
    ]
    ax3.legend(handles=custom_lines_2, loc='lower right', fontsize=FONTS['legend'], shadow=True)
    
    ax4.legend(handles=[Line2D([0], [0], color=COL['f_ring'], lw=LW['small'], ls=LS['f'], label=r'$\mathit{f_{22}}$')], loc='lower right', fontsize=FONTS['legend'], shadow=True)

    for ax in (ax1, ax2, ax3, ax4):
        ax.set_xlim(-10, 80)

    ax1.set_xticklabels([])
    ax2.set_xticklabels([])

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.subplots_adjust(hspace=0, wspace=0.27)

    if label is not None:
        plt.suptitle('RIT\\ %s' % (label,), fontsize=FONTS['labels']*1.2)
        # plt.savefig('global_fits/RIT_%s.pdf' % (label,), bbox_inches='tight')
    plt.show()
    plt.close()

### RIT

In [ ]:
df = pd.read_csv('../src/data/RIT_Parameters_non-spinning.csv')
df.set_index('ID', inplace=True)
masks = df['catalog'] == 'RIT'
rit_ids = df.index
chi = df.loc[rit_ids, 'chieff'].values
nu = df.loc[rit_ids, 'nu'].values
ecc = df.loc[rit_ids, 'ecc'].values
emrg = df.loc[rit_ids, 'Heff_til'].values
bmrg = df.loc[rit_ids, 'b_massless_EOB'].values
jmrg = df.loc[rit_ids, 'Jmrg_til'].values

In [ ]:
for id in rit_ids:
    res = pickle.load(open('../src/output/global_fits/nu_emrg_bmrg_3/RIT_%s/NR_sim.pkl' % (id,), 'rb'))
    res_bayes = pickle.load(open('../src/output/bayesian_global_fits/nu_emrg_bmrg_3/RIT_%s/NR_sim.pkl' % (id,), 'rb'))
    res_qc = pickle.load(open('../src/output/qc_fits_rit_non-spinning/RIT_%s_1/NR_sim.pkl' % (id,), 'rb'))
    wfs(res, res_bayes, res_qc, label='%s' % (id,))